# Examen ICCD753 Recuperación de Información

 Harry Guajan
 11/18/2026
## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado



| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |


## 3. Preparación e indexación del corpus NFCorpus

La unidad de indexación será cada documento completo, identificándolo por `_id`. No se dividirán los textos en fragmentos.

Este bloque de código básicamente se encarga de preparar y validar toda la base de datos antes de arrancar. Primero importa las librerías necesarias y se asegura de que estén descargadas las *stopwords* de NLTK para no tener errores más adelante.  luego busca el archivo `.parquet` dentro de la carpeta del corpus y lo carga en una tabla. Una vez cargado, revisa que no falten las columnas principales (`_id`, `title` y `text`), limpia los datos pasando todo a texto, rellenando los vacíos para que no se rompa el programa y verificando que no haya IDs duplicados, para finalmente imprimir el total de documentos y mostrar las primeras filas como confirmación de que todo está listo.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import pandas as pd
import nltk
from IPython.display import display
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

for recurso in ("stopwords",):
    try:
        nltk.data.find(f"corpora/{recurso}")
    except LookupError:
        nltk.download(recurso, quiet=True)

candidatos = [
    Path("corpus"),
    Path("EXAMENRI2BTAS") / "corpus",
]
CORPUS_DIR = next((ruta for ruta in candidatos if ruta.is_dir()), candidatos[0])
corpus_archivos = list(CORPUS_DIR.glob("*.parquet"))
if len(corpus_archivos) != 1:
    raise RuntimeError(f"Se esperaba un archivo Parquet en {CORPUS_DIR}, se encontraron {len(corpus_archivos)}.")

corpus = pd.read_parquet(corpus_archivos[0])
requeridas = {"_id", "title", "text"}
faltantes = requeridas.difference(corpus.columns)
if faltantes:
    raise KeyError(f"Faltan columnas obligatorias: {sorted(faltantes)}")

corpus = corpus[["_id", "title", "text"]].copy()
corpus["_id"] = corpus["_id"].astype(str)
corpus["title"] = corpus["title"].fillna("").astype(str)
corpus["text"] = corpus["text"].fillna("").astype(str)

if corpus["_id"].duplicated().any():
    duplicados = corpus.loc[corpus["_id"].duplicated(), "_id"].tolist()
    raise ValueError(f"Hay identificadores duplicados: {duplicados[:10]}")

print(f"Corpus: {len(corpus)} documentos")
display(corpus.head())

Corpus: 3633 documentos


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...
3,MED-301,Methylmercury: A Potential Environmental Risk ...,Epilepsy or seizure disorder is one of the mos...
4,MED-306,Sensitivity of Continuous Performance Test (CP...,Hit Reaction Time latencies (HRT) in the Conti...


### Decisiones de preprocesamiento

- **Normalización:** conversión a minúsculas.
- **Tokenización:** expresiones regulares sobre palabras alfanuméricas; se conservan palabras de al menos dos caracteres.
- **Stopwords:** eliminación con la lista inglesa de NLTK.
- **Stemming:** Snowball Stemmer para inglés.
- **Texto indexado:** título y texto completo del documento, sin fragmentación.

Este bloque hace   limpieza del texto y arma las estructuras principales del buscador para poder hacer consultas después. Primero une el título y el contenido de cada documento y los pasa por una función de preprocesamiento que convierte todo a minúsculas, saca palabras de al menos dos caracteres, elimina las *stopwords* en inglés y reduce cada término a su raíz usando *stemming*. 

Con esos textos ya limpios, construye el vocabulario completo y arma el índice invertido  además  calcula los pesos TF-IDF de todo el corpus en una matriz numérica lista para reutilizar. Al final, imprime estadísticas generales como el tamaño del vocabulario y la longitud promedio de los documentos, y muestra dos tablas de resumen para verificar que tanto los textos procesados como el índice de términos quedaron bien armados.

In [2]:
stopwords_ingles = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")
patron_tokens = re.compile(r"(?u)\b[a-z0-9][a-z0-9]+\b")

def preprocesar(texto):
    tokens = patron_tokens.findall(str(texto).lower())
    return [stemmer.stem(token) for token in tokens if token not in stopwords_ingles]

corpus["search_text"] = (
    corpus["title"].str.strip() + "\n" + corpus["text"].str.strip()
).str.strip()
corpus["tokens"] = corpus["search_text"].map(preprocesar)
corpus["processed_text"] = corpus["tokens"].map(lambda tokens: " ".join(tokens))

document_ids = corpus["_id"].tolist()
titles = corpus["title"].tolist()
original_texts = corpus["text"].tolist()
processed_texts = corpus["processed_text"].tolist()
token_lists = corpus["tokens"].tolist()

document_lengths = [len(tokens) for tokens in token_lists]
vocabulario = sorted({termino for tokens in token_lists for termino in set(tokens)})

# Índice invertido: termino -> [(posicion_documento, frecuencia)]
indice_invertido = defaultdict(list)
for doc_pos, tokens in enumerate(token_lists):
    frecuencias = Counter(tokens)
    for termino, frecuencia in frecuencias.items():
        indice_invertido[termino].append((doc_pos, frecuencia))

df = {termino: len(postings) for termino, postings in indice_invertido.items()}

# Relación estable entre cada posición interna y el identificador original.
id_por_posicion = {pos: doc_id for pos, doc_id in enumerate(document_ids)}
posicion_por_id = {doc_id: pos for pos, doc_id in id_por_posicion.items()}

# Representación TF-IDF para reutilizarla en el siguiente paso.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
    norm="l2",
)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_texts)

print("Documentos:", len(document_ids))
print("Vocabulario:", len(vocabulario))
print("Términos indexados:", len(indice_invertido))
print("Longitud mínima/media/máxima:",
      min(document_lengths),
      round(sum(document_lengths) / len(document_lengths), 2),
      max(document_lengths))
print("Forma de la matriz TF-IDF:", tfidf_matrix.shape)

tabla_corpus = pd.DataFrame({
    "Identificador": document_ids,
    "Título": titles,
    "Texto": original_texts,
    "Términos procesados": [len(tokens) for tokens in token_lists],
})
display(tabla_corpus.head(10))

tabla_vocabulario = pd.DataFrame([
    {"término": termino, "df": len(postings)}
    for termino, postings in sorted(indice_invertido.items())
])
display(tabla_vocabulario.head(20))

Documentos: 3633
Vocabulario: 18807
Términos indexados: 18807
Longitud mínima/media/máxima: 11 155.23 983
Forma de la matriz TF-IDF: (3633, 18807)


,Identificador,Título,Texto,Términos procesados
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an...",193
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...,181
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...,119
3,MED-301,Methylmercury: A Potential Environmental Risk ...,Epilepsy or seizure disorder is one of the mos...,125
4,MED-306,Sensitivity of Continuous Performance Test (CP...,Hit Reaction Time latencies (HRT) in the Conti...,188
5,MED-329,Phosphate is a vascular toxin.,Elevated phosphate (P) levels are seen in adva...,165
6,MED-330,Dietary Phosphorus Acutely Impairs Endothelial...,Excessive dietary phosphorus may increase card...,144
7,MED-332,Public health impact of dietary phosphorus exc...,This review explores the potential adverse imp...,178
8,MED-334,Differences among total and in vitro digestibl...,"OBJECTIVE: Among plant foods, grain products, ...",187
9,MED-335,Differences among total and in vitro digestibl...,OBJECTIVE: Meat and milk products are importan...,181


,término,df
0,00,91
1,000,136
2,000006073,1
3,00001,4
4,0001,117
5,0002,18
6,0003,9
7,0004,3
8,0005,8
9,0006,3


## 3.2. Modelos de recuperación

Se implementarán dos rankings independientes:

- **TF-IDF + similitud coseno:** compara el vector TF-IDF normalizado de la consulta con los vectores normalizados de los documentos.
- **BM25:** utiliza frecuencias documentales, frecuencia del término en el documento y normalización de longitud con $k_1=1.2$ y $b=0.75$.

Ambos métodos reciben texto libre, conservan los identificadores originales y pueden limitar el ranking mediante `top_k`.

Se implementan los dos motores de búsqueda. Primero defini una función auxiliar llamada `preparar_resultados` que recibe los puntajes calculados para todo el corpus, los ordena de mayor a menor, evita documentos repetidos y arma una lista limpia con los mejores *top-k* resultados (incluyendo su posición en el ranking, ID, puntaje, título y un fragmento del texto). 

Luego cree las dos funciones de búsqueda principales: 

- `buscar_tfidf`, que preprocesa la consulta, la convierte en un vector TF-IDF y calcula qué tanto se parece a cada documento usando similitud del coseno.

-  `buscar_bm25`, que recorre los términos limpios de la consulta en el índice invertido y va sumando la relevancia de cada documento aplicando la fórmula de BM25 (ajustando tanto por la frecuencia de las palabras como por qué tan largo es cada documento frente al promedio), para finalmente dejar ambas funciones listas para usar.

In [9]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

BM25_K1 = 1.2
BM25_B = 0.75

# Calcula la longitud promedio de los documentos procesados.
longitud_media = sum(document_lengths) / len(document_lengths)


def preparar_resultados(puntuaciones, top_k=10):
    """
    Ordena las puntuaciones de mayor a menor y convierte las posiciones
    internas en resultados con identificadores originales.
    """

    # Limita top_k al rango válido osea mínimo 1 y máximo la cantidad de documentos del corpus.
    
    top_k = max(1, min(int(top_k), len(document_ids)))


    orden = np.argsort(
        -np.asarray(puntuaciones, dtype=float),
        kind="stable"
    )

    resultados = []
    ids_vistos = set()

    for doc_pos in orden:
        doc_id = document_ids[int(doc_pos)]
        if doc_id in ids_vistos:
            continue

        ids_vistos.add(doc_id)

        # Construye el objeto que representa el documento recuperado.
        resultados.append({
            "rank": len(resultados) + 1,
            "document_id": doc_id,
            "score": float(puntuaciones[int(doc_pos)]),
            "title": titles[int(doc_pos)],
            "snippet": original_texts[int(doc_pos)][:300].strip(),
        })

        # Detiene el ranking cuando ya contiene top_k documentos.
        if len(resultados) == top_k:
            break
    return resultados


def buscar_tfidf(query, top_k=10):
    """
    Recupera documentos usando TF-IDF y similitud coseno.
    """
    tokens = preprocesar(query)

    # Convierte la consulta preprocesada en texto.
    processed_query = " ".join(tokens)
    # Convierte la consulta en un vector TF-IDF usando el mismo
    # TfidfVectorizer que se utilizó para crear tfidf_matrix.
    query_vector = tfidf_vectorizer.transform([processed_query])

    # Compara el vector de la consulta contra todos los vectores documentales.
    # ravel() convierte la matriz de resultados en un arreglo unidimensional.
    puntuaciones = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).ravel()

    return preparar_resultados(puntuaciones, top_k)


def buscar_bm25(query, top_k=10, k1=BM25_K1, b=BM25_B):
    """
    Recupera documentos utilizando el modelo BM25.
    """
    consulta_tokens = preprocesar(query)

    # Crea una puntuación inicial igual a cero para cada documento.
    puntuaciones = np.zeros(len(document_ids), dtype=float)

    # Cantidad total de documentos usados para calcular el IDF de BM25.
    total_documentos = len(document_ids)

    # set() elimina términos repetidos dentro de la misma consulta.
    for termino in set(consulta_tokens):

        # Obtiene los documentos que contienen el término.
        # Cada elemento tiene la forma (posición_del_documento, frecuencia).
        postings = indice_invertido.get(termino, [])

        # Si el término no existe en el vocabulario, se omite.
        if not postings:
            continue

        # Calcula cuántos documentos contienen el término.
        frecuencia_documental = len(postings)

        # Los términos que aparecen en pocos documentos reciben mayor peso.
        idf = np.log(
            1 +
            (
                total_documentos
                - frecuencia_documental
                + 0.5
            )
            /
            (
                frecuencia_documental
                + 0.5
            )
        )

        for doc_pos, frecuencia in postings:

            # Normaliza la frecuencia del término según la longitud  del documento y la longitud media del corpus.
            denominador = frecuencia + k1 * (
                1
                - b
                + b * document_lengths[doc_pos] / longitud_media
            )

            # Calcula cuánto aporta este término a la puntuación BM25 del documento actual.
    
            aporte = idf * (
                frecuencia * (k1 + 1)
            ) / denominador

            # Suma el aporte a la puntuación total del documento.
            puntuaciones[doc_pos] += aporte
    return preparar_resultados(puntuaciones, top_k)

print("Funciones TF-IDF y BM25 listas.")

Funciones TF-IDF y BM25 listas.


### Prueba reproducible de los rankings

La siguiente consulta pertenece a las consultas disponibles en el parquet de NFCorpus. Se muestran 10 resultados independientes para TF-IDF y BM25.

In [10]:
queries_candidatos = []
for ruta_queries in [Path("queries"), Path("EXAMENRI2BTAS") / "queries"]:
    if ruta_queries.is_dir():
        queries_candidatos = list(ruta_queries.glob("*.parquet"))
        break

if len(queries_candidatos) != 1:
    raise RuntimeError("No se encontró exactamente un archivo Parquet de consultas.")

queries = pd.read_parquet(queries_candidatos[0])
consulta_prueba = str(queries.iloc[1]["text"]) # Aqui podemos cambiar la consulta de prueba
#  Breast Cancer Cells Feed on Cholesterol es la primera consulta de prueba
print("Consulta:", consulta_prueba)

# Aqui consultamos a todos los documentos  con los dos  metodos
ranking_tfidf = buscar_tfidf(consulta_prueba, top_k=10)
ranking_bm25 = buscar_bm25(consulta_prueba, top_k=10)

# Solo para desplegar los resultados
display(pd.DataFrame(ranking_tfidf).drop(columns="snippet"))
display(pd.DataFrame(ranking_bm25).drop(columns="snippet"))

Consulta: Using Diet to Treat Asthma and Eczema


,rank,document_id,score,title
0,1,MED-2646,0.519689,"Do fast foods cause asthma, rhinoconjunctiviti..."
1,2,MED-2474,0.418161,The International Study of Asthma and Allergie...
2,3,MED-3708,0.400795,Is the asthma epidemic still ascending?
3,4,MED-2464,0.388264,Low vegetable intake is associated with allerg...
4,5,MED-2456,0.371641,"Diet and asthma, allergic rhinoconjunctivitis ..."
5,6,MED-2476,0.342398,"Diet, infection and wheezy illness: lessons fr..."
6,7,MED-2452,0.324644,A prospective study of diet and adult-onset as...
7,8,MED-4551,0.323534,Maternal meat and fat consumption during pregn...
8,9,MED-2484,0.321797,The burden of childhood asthma
9,10,MED-5072,0.268884,Lycopene-rich treatments modify noneosinophili...


,rank,document_id,score,title
0,1,MED-2646,18.765231,"Do fast foods cause asthma, rhinoconjunctiviti..."
1,2,MED-2456,18.399079,"Diet and asthma, allergic rhinoconjunctivitis ..."
2,3,MED-2474,17.599687,The International Study of Asthma and Allergie...
3,4,MED-2476,17.149617,"Diet, infection and wheezy illness: lessons fr..."
4,5,MED-2471,15.362096,Which population level environmental factors a...
5,6,MED-3685,15.217847,Caesarean delivery and risk of atopy and aller...
6,7,MED-4548,13.681171,Common Household Chemicals and the Allergy Ris...
7,8,MED-2649,12.374178,Dietary meat and fat intake and prevalence of ...
8,9,MED-4551,12.217334,Maternal meat and fat consumption during pregn...
9,10,MED-2464,11.460528,Low vegetable intake is associated with allerg...


## 3.3. Servicio web

El servicio expone dos endpoints:

- `GET /health`: confirma que la API está disponible.
- `POST /search`: recibe una consulta JSON y devuelve el ranking solicitado con TF-IDF o BM25.

Los resultados se generan reutilizando las funciones `buscar_tfidf` y `buscar_bm25` definidas en la sección anterior.

##### Nota sobre el despliegue en Internet

Para cumplir con el requisito de publicación en Internet sin depender de la ejecución local de este notebook, esta misma aplicación fue extraída y desplegada como un servicio web independiente en **Render**, accesible públicamente a través de las siguientes rutas:

- **URL base del servicio:** `https://recuperacion-de-la-infromacion-2026b.onrender.com`
- **Comprobación de disponibilidad (`GET`):** `https://recuperacion-de-la-infromacion-2026b.onrender.com/health`
- **Endpoint de búsqueda (`POST` con cuerpo JSON):** `https://recuperacion-de-la-infromacion-2026b.onrender.com/search`

Unicamente se encuentran habilitados `/health` y `/search`. El endpoint `/search` requiere una solicitud `POST` con encabezado `Content-Type: application/json`, por lo que no se consulta directamente desde la barra del navegador mediante `GET`, sino a través de un cliente HTTP como Postman con un cuerpo JSON similar al siguiente (admitiendo `"bm25"` o `"tfidf"` en el campo `model`):

```json
{
  "query": "using diet to treat asthma",
  "model": "bm25",
  "top_k": 10
}

In [11]:
from flask import Flask, jsonify, request

# Crea la aplicación web y permite que este notebook funcione como servidor.
app = Flask(__name__)


@app.get("/health")
def health():
    """Informa si el servicio está disponible."""
    return jsonify({
        "status": "ok",
        "corpus": "NFCorpus",
        "documents": len(document_ids)
    })


@app.post("/search")
def search():
    """Valida la solicitud y devuelve uno de los dos rankings."""
    body = request.get_json(silent=True)
    if not isinstance(body, dict):
        return jsonify({"error": "El cuerpo debe ser un objeto JSON."}), 400

    query = body.get("query")
    model = body.get("model")
    top_k = body.get("top_k", 10)

    if not isinstance(query, str) or not query.strip():
        return jsonify({"error": "El campo query debe ser texto y no puede estar vacío."}), 400

    if model not in ("tfidf", "bm25"):
        return jsonify({"error": "El campo model debe ser 'tfidf' o 'bm25'."}), 400

    if isinstance(top_k, bool) or not isinstance(top_k, (int, float)):
        return jsonify({"error": "El campo top_k debe ser un entero positivo."}), 400

    top_k = int(top_k)
    if top_k < 1:
        return jsonify({"error": "El campo top_k debe ser un entero positivo."}), 400

    if model == "tfidf":
        results = buscar_tfidf(query.strip(), top_k=top_k)
    else:
        results = buscar_bm25(query.strip(), top_k=top_k)

    return jsonify({
        "model": model,
        "results": results
    })


print("API Flask creada: GET /health y POST /search")

API Flask creada: GET /health y POST /search


### Prueba local de los endpoints

Antes de publicar el servicio se usa el cliente de prueba de Flask. Así se valida el formato JSON sin abrir un servidor externo.

In [12]:
with app.test_client() as client:
    respuesta_health = client.get("/health")
    print("GET /health:", respuesta_health.status_code, respuesta_health.get_json())

    respuesta_tfidf = client.post(
        "/search",
        json={"query": "using diet to treat asthma", "model": "tfidf", "top_k": 3}
    )
    print("POST /search TF-IDF:", respuesta_tfidf.status_code)
    print(respuesta_tfidf.get_json())

    respuesta_bm25 = client.post(
        "/search",
        json={"query": "using diet to treat asthma", "model": "bm25", "top_k": 3}
    )
    print("POST /search BM25:", respuesta_bm25.status_code)
    print(respuesta_bm25.get_json())

    respuesta_error = client.post("/search", json={"query": "", "model": "bm25"})
    print("Error esperado:", respuesta_error.status_code, respuesta_error.get_json())

GET /health: 200 {'corpus': 'NFCorpus', 'documents': 3633, 'status': 'ok'}
POST /search TF-IDF: 200
{'model': 'tfidf', 'results': [{'document_id': 'MED-3708', 'rank': 1, 'score': 0.5522359444017465, 'snippet': 'Asthma is a phenotypically heterogeneous disorder of multifactorial origins that affects 300 million people suffering from asthma and more than 250,000 asthma-related deaths each year. Although treatment for asthma has improved, its prevalence continues to increase, particularly in low and middle in', 'title': 'Is the asthma epidemic still ascending?'}, {'document_id': 'MED-2464', 'rank': 2, 'score': 0.5349695205195418, 'snippet': "BACKGROUND: In recent decades, children's diet quality has changed and asthma prevalence has increased, although it remains unclear if these events are associated. OBJECTIVE: To examine children's total and component diet quality and asthma and airway hyperresponsiveness (AHR), a proxy for asthma se", 'title': 'Low vegetable intake is associated with 

Para iniciar el servicio localmente, ejecuta la siguiente celda y abre `http://127.0.0.1:5000/health`. Mantén esta celda ejecutándose mientras pruebes el servicio con Postman.

Después se podrá desplegar esta aplicación en Render, Railway u otra plataforma pública.

In [ ]:
# Ejecuta manualmente esta celda para iniciar el servidor local.
#app.run(host="127.0.0.1", port=5000, debug=False)
print("Descomenta app.run(...) para iniciar el servicio local.")

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [18/Nov/2026 12:02:20] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Nov/2026 12:02:21] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [18/Nov/2026 12:02:25] "GET /health HTTP/1.1" 200 -


Descomenta app.run(...) para iniciar el servicio local.


## 4. Evaluación de la recuperación

Se evaluarán TF-IDF y BM25 con las consultas de la partición `test` y sus juicios de relevancia. Para cada consulta se calcularán:

- **Precision@10:** proporción de documentos relevantes entre los 10 recuperados.
- **Recall@10:** proporción de documentos relevantes recuperados entre todos los relevantes de la consulta.
- **NDCG@10:** calidad del ranking considerando la posición y el nivel de relevancia declarado en los qrels.

La métrica final será el promedio de los resultados sobre todas las consultas `test` disponibles.

In [16]:
import math

# Carga los qrels de prueba.
qrels_directorio = None
for ruta in [Path("qrels"), Path("EXAMENRI2BTAS") / "qrels"]:
    if ruta.is_dir():
        qrels_directorio = ruta
        break

if qrels_directorio is None:
    raise RuntimeError("No se encontró la carpeta qrels.")

test_qrels = pd.read_csv(
    qrels_directorio / "test.tsv",
    sep="\t",
    dtype={"query-id": str, "corpus-id": str, "score": int},
)

# Cada consulta test puede tener varios documentos relevantes.
relevancia_test = {}
for query_id, document_id, score in test_qrels.itertuples(index=False, name=None):
    query_id = str(query_id)
    document_id = str(document_id)
    relevancia_test.setdefault(query_id, {})[document_id] = int(score)

# Conecta los IDs de las consultas con su texto.
textos_query = {
    str(fila["_id"]): str(fila["text"])
    for _, fila in queries.iterrows()
}

query_ids_test = sorted(relevancia_test)
query_ids_test = [qid for qid in query_ids_test if qid in textos_query]

print("Consultas test disponibles:", len(query_ids_test))
print("Consultas test mostradas:", query_ids_test[:10])


def precision_recall_ndcg(ranking, relevancias, k=10):
    """Calcula P@k, Recall@k y NDCG@k para un ranking."""
    ranking_k = ranking[:k]
    ids_recuperados = [resultado["document_id"] for resultado in ranking_k]

    relevantes = {
        document_id
        for document_id, score in relevancias.items()
        if score > 0
    }

    relevantes_recuperados = len(set(ids_recuperados) & relevantes)
    precision = relevantes_recuperados / k
    recall = relevantes_recuperados / len(relevantes) if relevantes else 0.0

    dcg = 0.0
    for posicion, document_id in enumerate(ids_recuperados, start=1):
        relevancia = relevancias.get(document_id, 0)
        if relevancia > 0:
            dcg += (2 ** relevancia - 1) / math.log2(posicion + 1)

    grados_ideales = sorted(relevancias.values(), reverse=True)[:k]
    idcg = 0.0
    for posicion, relevancia in enumerate(grados_ideales, start=1):
        if relevancia > 0:
            idcg += (2 ** relevancia - 1) / math.log2(posicion + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0
    return precision, recall, ndcg


metricas = {
    "tfidf": [],
    "bm25": [],
}

for indice, query_id in enumerate(query_ids_test, start=1):
    consulta = textos_query[query_id]
    relevancias = relevancia_test[query_id]

    ranking_tfidf = buscar_tfidf(consulta, top_k=10)
    ranking_bm25 = buscar_bm25(consulta, top_k=10)

    metricas["tfidf"].append(
        precision_recall_ndcg(ranking_tfidf, relevancias, k=10)
    )
    metricas["bm25"].append(
        precision_recall_ndcg(ranking_bm25, relevancias, k=10)
    )

    if indice % 50 == 0:
        print("Consultas evaluadas:", indice)


tabla_metricas = pd.DataFrame([
    {
    "Modelo": "TF-IDF",
    "Precision@10": sum(m[0] for m in metricas["tfidf"]) / len(metricas["tfidf"]),
    "Recall@10": sum(m[1] for m in metricas["tfidf"]) / len(metricas["tfidf"]),
    "NDCG@10": sum(m[2] for m in metricas["tfidf"]) / len(metricas["tfidf"]),
    },
    {
    "Modelo": "BM25",
    "Precision@10": sum(m[0] for m in metricas["bm25"]) / len(metricas["bm25"]),
    "Recall@10": sum(m[1] for m in metricas["bm25"]) / len(metricas["bm25"]),
    "NDCG@10": sum(m[2] for m in metricas["bm25"]) / len(metricas["bm25"]),
    },
])

tabla_metricas.iloc[:, 1:] = tabla_metricas.iloc[:, 1:].round(6)
display(tabla_metricas)

Consultas test disponibles: 323
Consultas test mostradas: ['PLAIN-1008', 'PLAIN-1018', 'PLAIN-102', 'PLAIN-1028', 'PLAIN-1039', 'PLAIN-1050', 'PLAIN-1066', 'PLAIN-1088', 'PLAIN-1098', 'PLAIN-1109']
Consultas evaluadas: 50
Consultas evaluadas: 100
Consultas evaluadas: 150
Consultas evaluadas: 200
Consultas evaluadas: 250
Consultas evaluadas: 300


,Modelo,Precision@10,Recall@10,NDCG@10
0,TF-IDF,0.227245,0.148824,0.308518
1,BM25,0.238390,0.151116,0.327527


### Interpretación de la comparación

Sobre las 323 consultas de la partición `test`, los resultados ejecutados fueron:

| Modelo | Precision@10 | Recall@10 | NDCG@10 |
|---|---:|---:|---:|
| TF-IDF | 0.227245 | 0.148824 | 0.308518 |
| BM25 | 0.238390 | 0.151116 | 0.327527 |

BM25 obtuvo mejores valores en las tres métricas. Su ventaja fue mayor en NDCG@10 (+0.019009), lo que indica que ordenó mejor los documentos relevantes. BM25 también obtuvo una Precision@10 mayor (+0.011145) y un Recall@10 ligeramente superior (+0.002292).